In [1]:

from google.colab import drive
drive.mount('/content/drive')

import sys, importlib
sys.path.append("/content/drive/MyDrive/ft_vs_rag_project")

from ft_vs_rag_improved_pipeline import (
    DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC,
    hotpot_extract, save_jsonl, make_dynamic_split,
    build_faiss_index_from_jsonl
)

print("OK ✅", DATA_DIR)



Mounted at /content/drive
OK ✅ /content/ft_vs_rag_multidata/data_hotpot


# Indexing the Docs

In [2]:
# ===========================
# GPU: embeddings & indexing
# ===========================
# Runtime ➜ GPU (T4/L4/A100). Do NOT use TPU for sentence-transformers/FAISS.

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ft_vs_rag_project"
DATA_ROOT = f"{DRIVE_PROJECT_DIR}/ft_vs_rag_multidata"  # where CPU notebook synced data
!test -d "$DATA_ROOT" || (echo "Missing $DATA_ROOT, please run the CPU notebook first." && false)

# -----------------------------------
# Installs (GPU-friendly)
# -----------------------------------
#!pip install -q sentence-transformers faiss-gpu  # or faiss-cpu if GPU FAISS not available
!pip install -q sentence-transformers faiss-cpu

# If Axolotl is used later for training:
# !pip install -q axolotl[deepspeed]  # only if you plan to launch training here


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 111.0 MB/s eta 0:00:00


In [3]:

# -----------------------------------
# Imports
# -----------------------------------
from pathlib import Path
from ft_vs_rag_improved_pipeline import (
    DATA_DIR, INDEX_DIR_FULL, INDEX_DIR_DYNAMIC,
    build_faiss_index_from_jsonl, create_qa_finetune_dataset,
    write_axolotl_configs
)

# Ensure our module writes/reads from the same CPU-prepared folder
print("Module DATA_DIR:", DATA_DIR)
print("Drive data root:", DATA_ROOT)

# -----------------------------------
# Use a faster or stronger embedding model (GPU can handle larger models)
# -----------------------------------
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"  # good quality on GPU
# For even faster runs (slight quality drop):
# EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# -----------------------------------
# Build FAISS indexes on GPU
#   FULL    = stable + dynamic_indexed
#   DYNAMIC = dynamic_indexed only
# -----------------------------------
full_sources = [
    Path(DATA_ROOT) / "data_hotpot" / "stable_docs.jsonl",
    Path(DATA_ROOT) / "data_hotpot" / "dynamic_indexed_docs.jsonl"
]
dyn_sources = [
    Path(DATA_ROOT) / "data_hotpot" / "dynamic_indexed_docs.jsonl"
]

# Make sure module reads those same files (they are under /content/drive/...)
# build into default index dirs (under /content/...) for simplicity
built_full = build_faiss_index_from_jsonl(
    INDEX_DIR_FULL, full_sources, encoder_model=EMBED_MODEL
)
built_dyn  = build_faiss_index_from_jsonl(
    INDEX_DIR_DYNAMIC, dyn_sources, encoder_model=EMBED_MODEL
)

print("Built FULL index ->", built_full)
print("Built DYNAMIC index ->", built_dyn)

# -----------------------------------
# Create QA fine-tune dataset WITH contexts (uses embeddings on GPU)
#   Reads train_ft_qas.jsonl + stable_docs.jsonl (as retrieval source),
#   and writes docs_qa_with_context.jsonl for SFT.
#   If you want retrieval over both stable+dynamic for SFT, replace docs_filename accordingly.
# -----------------------------------
qa_path = create_qa_finetune_dataset(
    data_dir=Path(DATA_ROOT) / "data_hotpot",
    docs_filename="stable_docs.jsonl",     # SFT on stable only (common in hybrid setup)
    qas_filename="train_ft_qas.jsonl",
    output_filename="docs_qa_with_context.jsonl",
    encoder_model=EMBED_MODEL,
    top_k=3
)
print("Wrote:", qa_path)

# -----------------------------------
# (Optional) write Axolotl configs (pretrain + SFT)
# -----------------------------------
cfgs = write_axolotl_configs(
    base_model="mistralai/Mistral-7B-Instruct-v0.2",
    pretrain_jsonl=Path(DATA_ROOT) / "data_hotpot" / "docs_for_pretrain.jsonl",
    qa_jsonl=Path(DATA_ROOT) / "data_hotpot" / "docs_qa_with_context.jsonl",
)
print("Config files:", cfgs)

# -----------------------------------
# Sanity: list results
# -----------------------------------
!ls -lh "/content/ft_vs_rag_multidata/"
!ls -lh "/content/ft_vs_rag_multidata/faiss_index_full"
!ls -lh "/content/ft_vs_rag_multidata/faiss_index_dynamic"
!ls -lh "{DATA_ROOT}/data_hotpot" | sed -n '1,200p'


Module DATA_DIR: /content/ft_vs_rag_multidata/data_hotpot
Drive data root: /content/drive/MyDrive/ft_vs_rag_project/ft_vs_rag_multidata


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/17305 [00:00<?, ?it/s]

Batches:   0%|          | 0/637 [00:00<?, ?it/s]

Built FULL index -> /content/ft_vs_rag_multidata/faiss_index_full/index.faiss
Built DYNAMIC index -> /content/ft_vs_rag_multidata/faiss_index_dynamic/index.faiss


KeyboardInterrupt: 

In [ ]:
# Save index files in Google Drive (or manually download files from LOCAL_DIR and upload them to Google Drive at DRIVE_TARGET)

# local source (in Colab)
LOCAL_DIR = "/content/ft_vs_rag_multidata"

# target folder on Google Drive (change the path if you want a different location)
DRIVE_TARGET = DRIVE_PROJECT_DIR + "/ft_vs_rag_multidata"

# create target folder if not exists
!mkdir -p "$DRIVE_TARGET"

# copy (sync) everything, preserving directory structure
!rsync -ah --progress "$LOCAL_DIR/" "$DRIVE_TARGET/"
print("✅ Sync complete.")


NameError: name 'DRIVE_PROJECT_DIR' is not defined